# INPP5D locus across all AD GWAS studies (data processing)

The INPP5D locus (chr2) is a SuSiE-rRSS power-drop example in Fig S12c (Bellenguez EADB 2022, `chr2_231326994_235132526`): SuSiE-rRSS reports no credible set at a signal SuSiE-RSS resolves. Here the same locus in **every** study: for each study, the fine-mapping block holding the INPP5D TSS (`gene_cis_TADB_mapper.tsv`), preferring the Fig S12c block when the study has a fit there. `load_locus()` (`R/utils.R`) loads the GWAS −log10 p, SuSiE-RSS and SuSiE-rRSS PIPs / CSs and the AD xQTL evidence, plus the 2026 AD GWAS colocalization rows.

SuSiE-rRSS is the full-pipeline result: the SER model replaces the fit when the SER fallback was used (`serFallbackUsed` in `export/SER_flag.tsv`, else the fit's own reliability flag), as in `AD_GWAS_Gene_Loci_All_Studies`.

**Outputs:**
- `figure_data/AD_GWAS_INPP5D_All_Studies.rds`: `list(gene, tss, loci)`, `loci` named by study (each = `load_locus()` output + `tm` + `ser_fallback`)
- `figure_data/AD_GWAS_INPP5D_CS_Counts.csv`: study × block: credible sets under each method, SER fallback
- `figure_data/AD_GWAS_INPP5D_CS_PIP_Support.csv`: every credible set of either method, the other method's PIP sum over its variants and whether the other method reports a set there (`cs_pip_support()`)

In [1]:
source("../../R/utils.R")
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)   # getSumStatsDf()
library(readxl)
filter <- dplyr::filter; select <- dplyr::select

gene             <- "INPP5D"
prefer_block     <- "chr2_231326994_235132526"     # Fig S12c power-drop block (Bellenguez EADB 2022)
gene_mapper_file <- "/hpc/users/sunh17/haochen/rss_qc/gene_cis_TADB_mapper.tsv"

root      <- "/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10"
susie_sub <- "AD_GWAS_rss_baseline_r4_L10"
ebmix_sub <- "AD_GWAS_rss_ebmix_r4_L10"

### AD GWAS explorer data, 2026 AD GWAS colocalization, SER fallback flags
AD_loci <- suppressMessages(suppressWarnings(
  read_excel("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/unified_AD_loci_xQTL_summary.xlsx", skip = 2)))
transmap_long <- read_tsv(file.path(root, ebmix_sub, "export/transmap_AD_target_results_pv1e4.tsv"),
                          show_col_types = FALSE) %>% tidy_transmap()
ser_flag <- read_tsv(file.path(root, ebmix_sub, "export/SER_flag.tsv"), show_col_types = FALSE) %>%
  transmute(study, block_id, ser_fallback = serFallbackUsed %in% TRUE)

ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”


## Study → block holding the INPP5D TSS

In [2]:
gene_pos <- read_tsv(gene_mapper_file, show_col_types = FALSE) %>%
  rename(chr = `#chr`) %>% filter(gene_name == gene) %>%
  transmute(gene = gene_name, chr = str_remove(chr, "^chr"), tss = TSS) %>% distinct(gene, .keep_all = TRUE)
gene_pos
stopifnot(nrow(gene_pos) == 1)

## every study x block with a SuSiE-rRSS fit whose block holds the TSS
fits <- tibble(unit = list.files(file.path(root, ebmix_sub, "output/fine_mapping"), pattern = "\\.gwas_finemap\\.rds$")) %>%
  mutate(unit     = str_remove(unit, "\\.gwas_finemap\\.rds$"),
         study    = str_split_fixed(unit, "\\.", 2)[, 1],
         block_id = str_split_fixed(unit, "\\.", 2)[, 2],
         m = str_match(block_id, "^chr([0-9XY]+)_([0-9]+)_([0-9]+)$"),
         chr = m[, 2], bstart = as.numeric(m[, 3]), bend = as.numeric(m[, 4])) %>%
  select(-m) %>%
  filter(chr == gene_pos$chr, bstart <= gene_pos$tss, gene_pos$tss <= bend)
fits %>% select(study, block_id) %>% as.data.frame()       # all candidate blocks

## one block per study: the Fig S12c block when fitted, else the narrowest block holding the TSS
study_block <- fits %>%
  group_by(study) %>% arrange(block_id != prefer_block, bend - bstart, .by_group = TRUE) %>% slice(1) %>% ungroup()
study_block %>% select(study, block_id) %>% as.data.frame()

gene,chr,tss
<chr>,<chr>,<dbl>
INPP5D,2,233059966


study,block_id
<chr>,<chr>
AD_Bellenguez_2022,chr2_231326994_233100399
AD_Bellenguez_2022,chr2_231326994_235132526
AD_Bellenguez_EADB_2022,chr2_231326994_233100399
AD_Bellenguez_EADB_2022,chr2_231326994_235132526
AD_Bellenguez_EADI_2022,chr2_231326994_233100399
AD_Bellenguez_EADI_2022,chr2_231326994_235132526
AD_Bellenguez_GRACE_2022,chr2_231326994_233100399
AD_Bellenguez_GRACE_2022,chr2_231326994_235132526
AD_Bellenguez_main_2026,chr2_231326994_233100399


study,block_id
<chr>,<chr>
AD_Bellenguez_2022,chr2_231326994_235132526
AD_Bellenguez_EADB_2022,chr2_231326994_235132526
AD_Bellenguez_EADI_2022,chr2_231326994_235132526
AD_Bellenguez_GRACE_2022,chr2_231326994_235132526
AD_Bellenguez_main_2026,chr2_231326994_235132526
AD_Bellenguez_nobiobank_2026,chr2_231326994_235132526
AD_Bellenguez_noproxy_2026,chr2_231326994_235132526
AD_Jansen_2021,chr2_231326994_235132526
AD_Kunkle_Stage1_2019,chr2_231326994_235132526


## Loci

In [3]:
get_fit <- function(sub, sid, bid)
  attributes(readRDS(file.path(root, sub, "output/fine_mapping", paste0(sid, ".", bid, ".gwas_finemap.rds"))))$elementMetadata$susieFit[[1]]

## SuSiE-rRSS pipeline result: the SER model replaces the fit when the fallback was used
rrss_pipeline <- function(loc) {
  fit  <- get_fit(ebmix_sub, loc$sid, loc$bid)
  fl   <- filter(ser_flag, study == loc$sid, block_id == loc$bid)
  flag <- if (nrow(fl)) any(fl$ser_fallback) else isTRUE(fit$R_finite_diagnostics$R_reliability_flag)
  ser  <- fit$R_finite_diagnostics$ser_model
  if (flag && !is.null(ser)) {
    if (is.null(dim(ser$alpha))) ser$alpha <- matrix(ser$alpha, nrow = 1)
    colnames(ser$alpha) <- colnames(fit$alpha)
    loc$pip_df <- bind_rows(filter(loc$pip_df, method != "SuSiE-rRSS"), locus_tidy_fit(ser, "SuSiE-rRSS"))
  }
  loc$ser_fallback <- flag
  loc
}

loci <- pmap(study_block, function(study, block_id, ...)
  tryCatch(load_locus(block_id, study, AD_loci, root = root, susie_sub = susie_sub, ebmix_sub = ebmix_sub) %>%
             rrss_pipeline() %>% { .$tm <- locus_transmap(transmap_long, .$reg); . },
           error = function(e) { message("FAILED: ", study, ".", block_id, " : ", conditionMessage(e)); NULL })) %>%
  set_names(study_block$study) %>% compact()
length(loci)                                  # studies loaded

FAILED: AD_Bellenguez_EADI_2022.chr2_231326994_235132526 : ℹ In argument: `chr == as.character(reg$chr)`.
Caused by error in `mutate()`:
ℹ In argument: `logp = -(pnorm(-abs(z), log.p = TRUE) +
  log(2))/log(10)`.
Caused by error:
! object 'z' not found



[1] 11

In [4]:
## credible sets per study and method, SER fallback
cs_counts <- imap_dfr(loci, function(loc, sid)
  loc$pip_df %>% filter(cs != "0") %>% distinct(method, cs) %>% count(method) %>%
    complete(method = c("SuSiE-RSS", "SuSiE-rRSS"), fill = list(n = 0L)) %>%
    pivot_wider(names_from = method, values_from = n) %>%
    mutate(study = sid, study_label = unname(cohort_labels[sid]), block_id = loc$bid, ser_fallback = loc$ser_fallback, .before = 1))
as.data.frame(cs_counts)

## every credible set of either method: the other method's PIP sum over its variants
cs_support <- map_dfr(loci, cs_pip_support)
cs_support %>% filter(!other_in_cs) %>% as.data.frame()      # sets the other method did not report

save_data(list(gene = gene, tss = gene_pos$tss, loci = loci), "AD_GWAS_INPP5D_All_Studies")
write_csv(cs_counts,  file.path(data_dir, "AD_GWAS_INPP5D_CS_Counts.csv"))
write_csv(cs_support, file.path(data_dir, "AD_GWAS_INPP5D_CS_PIP_Support.csv"))

study,study_label,block_id,ser_fallback,SuSiE-rRSS,SuSiE-RSS
<chr>,<chr>,<chr>,<lgl>,<int>,<int>
AD_Bellenguez_2022,Bell.,chr2_231326994_235132526,FALSE,1,1
AD_Bellenguez_EADB_2022,Bell. EADB,chr2_231326994_235132526,FALSE,0,1
AD_Bellenguez_GRACE_2022,Bell. GRACE,chr2_231326994_235132526,FALSE,0,0
AD_Bellenguez_main_2026,EADB. 2026,chr2_231326994_235132526,FALSE,1,1
AD_Bellenguez_nobiobank_2026,EADB. 2026 -biobank,chr2_231326994_235132526,FALSE,1,1
AD_Bellenguez_noproxy_2026,EADB. 2026 -proxy,chr2_231326994_235132526,FALSE,1,1
AD_Jansen_2021,Jansen,chr2_231326994_235132526,FALSE,0,0
AD_Kunkle_Stage1_2019,Kunkle,chr2_231326994_235132526,FALSE,0,0
AD_Wightman_Excluding23andMe_2021,Wight. -23,chr2_231326994_235132526,FALSE,0,0


sid,bid,method,cs,n_variants,pip_sum,other,other_pip_sum,other_in_cs
<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<chr>,<dbl>,<lgl>
AD_Bellenguez_EADB_2022,chr2_231326994_235132526,SuSiE-RSS,1,1,0.9532832,SuSiE-rRSS,0.9443119,FALSE
